# Suite LST — Lists

A property whose schema is an `OfIndexed` holds a list: ordered items, each a value of one item schema, read as a
frozen array and written over the wire as an array. The value objects in a list belong to the list's owner: they have
identities, may have adjacencies, are copied and removed with their owner, and are linked by `$id`.

In [ ]:
import { Comparison as C, JSON, Plain, Proxies, Reachable, Schemas as S, Validators, YAML } from "@mbse/schemas/Framework";
import { AttributeError, DecodeError, LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import type { Visitable } from "@mbse/schemas/Framework/Visitors";
import { assert, equal, map, raises, same, text } from "./support.js";

const store = new Proxies.OfStore();

/** An `OfAny` Spec for a list of `spec`. */
function list_of(spec: S.OfAny.Spec) {
  return (t: S.OfAny.Builder) => t.as_indexed((i) => i.of(spec));
}

/** An `OfAny` Spec for a list of `spec` keyed by `key`. */
function keyed(key: S.OfAny.Spec, spec: S.OfAny.Spec) {
  return (t: S.OfAny.Builder) => t.as_indexed((i) => i.key(key).of(spec));
}

const Str = (t: S.OfAny.Builder) => t.as_native(String);
const Int = (t: S.OfAny.Builder) => t.as_native(BigInt);
const Port = new S.OfObject.Builder().properties(text("name")).create();
const Wire = new S.OfRelation.Builder().links("source", "sink").create();
const Float = (t: S.OfAny.Builder) => t.as_native(Number);
const Cell = new S.OfObject.Builder().properties(text("r", BigInt), text("c", BigInt), text("w", Number),
  (p) => p.name("path").of(list_of(Int)), (p) => p.name("meta").of(keyed(Str, Str))).create();
const Rack = new S.OfObject.Builder().properties(text("name"), (p) => p.name("port").of(Port)).create();
const Shape = new S.OfUnion.Builder().branches((b) => b.name("circle").of((t) => t.as_native(Number)),
  (b) => b.name("label").of(Str)).create();
const Part = new S.OfObject.Builder().name("Part").ref().properties(
  text("name"), (p) => p.name("tags").of(list_of(Str)), (p) => p.name("ports").of(list_of(Port)),
  (p) => p.name("grid").of(list_of(list_of(Int))), (p) => p.name("banks").of(list_of(list_of(Port))),
  (p) => p.name("shapes").of(list_of(Shape)), (p) => p.name("racks").of(list_of(Rack)),
  (p) => p.name("row").of((t) => t.as_indexed((i) => i.of(Int).extent({ minimum: 1n, maximum: 3n }))),
  (p) => p.name("attrs").of(keyed(Str, Float)), (p) => p.name("weights").of(keyed(Float, Str)),
  (p) => p.name("flags").of(keyed((t) => t.as_native(Boolean), Str)),
  (p) => p.name("named").of(keyed((t) => t.as_native(Uint8Array), Port)),
  (p) => p.name("cells").of(keyed(Cell, Int))).create();
const Hub = new S.OfObject.Builder().name("Hub").ref().properties(text("name")).relations(
  (r) => r.name("feeds").of(Wire).me("sink")).create();
new S.OfObject.Builder(Port).relations((r) => r.name("wires").of(Wire).me("source"),
  (r) => r.name("fed").of(Wire).me("sink")).update();
const Labels = new S.OfRelation.Builder().links("part", "hub").properties((p) => p.name("labels").of(list_of(Str)),
  (p) => p.name("notes").of(keyed(Str, Str))).create();
new S.OfObject.Builder(Part).relations((r) => r.name("labelled").of(Labels).me("part")).update();
new S.OfObject.Builder(Hub).relations((r) => r.name("labels").of(Labels).me("hub")).update();
store.register(Part);
store.register(Hub);
const B = store;

/** A `Visitable` Part whose `accept` runs `write(visitor)`: data that proxies do not produce. */
class Written implements Visitable {
  constructor(private readonly write: (visitor: any) => void) {}

  identity(): unknown {
    return this;
  }

  schema_name(): string {
    return "Part";
  }

  owner(): null {
    return null;
  }

  accept(visitor: any): void {
    this.write(visitor);
  }
}

/** A one-object Part snapshot with `values`. */
function snapshot(values: Record<string, unknown>): unknown {
  return map({ root: "s0", objects: { s0: values } });
}

function decode_error(plain: unknown, match: string): void {
  raises(DecodeError, () => Plain.FromPlain(B)(Part, plain), match);
}

/** The root object of a snapshot. */
function s0(plain: any): any {
  return plain.get("objects").get("s0");
}

## LST-01 · `OfIndexed` holds one item schema of any kind, validated, and never a reference object schema

In [ ]:
{
  assert([Part, Port, Hub, Labels].every((schema) => schema.validate().length === 0));
  const Tags = new S.OfIndexed.Builder().of(Str).create();
  assert(Tags instanceof S.OfIndexed.Data && Tags.item instanceof S.OfNative.Data && Tags.validate().length === 0);
  assert(new S.OfAny.Builder().as_indexed(Tags).create() === Tags && S.OfIndexed.resolve(Tags) === Tags);
  assert((Part.properties.get("grid")?.type as S.OfIndexed.Data).item instanceof S.OfIndexed.Data); // lists of lists
  assert((Part.properties.get("shapes")?.type as S.OfIndexed.Data).item instanceof S.OfUnion.Data);
  // The item's own problems are the list's.
  assert(same(new S.OfIndexed.Data().validate(), ["item: not a schema: None"]));
  const Wide = new S.OfIndexed.Builder().of((t) => t.as_native((n) => n.type(BigInt).bits(0n))).create();
  assert(same(Wide.validate(), ["item: a width in bits must be a positive int or a term, got 0"]));
  const Holder = new S.OfObject.Builder().properties((p) => p.name("wide").of(Wide)).create();
  assert(same(Holder.validate(), ["property 'wide': item: a width in bits must be a positive int or a term, got 0"]));
  // A reference object schema is never an item's, at any depth, nor through a union.
  const Card = new S.OfObject.Builder().ref().create();
  const Either = new S.OfUnion.Builder().branches((b) => b.name("cards").of(list_of(Card)),
    (b) => b.name("none").of(list_of(Int))).create();
  const Cards = new S.OfObject.Builder().properties((p) => p.name("cards").of(list_of(Card)),
    (p) => p.name("deep").of(list_of(list_of(Card))), (p) => p.name("either").of(Either)).create();
  assert(same(Cards.validate(), ["property 'cards': a reference object schema cannot be a property's type",
    "property 'deep': a reference object schema cannot be a property's type",
    "property 'either': a reference object schema cannot be a property's type"]));
  // An entry's value objects have no adjacencies, in lists too.
  const Pair = new S.OfRelation.Builder().links("a", "b").properties((p) => p.name("ports").of(list_of(Port))).create();
  assert(same(Pair.validate(), ["property 'ports': a value object held by an entry cannot have adjacencies"]));
  const Pin = new S.OfObject.Builder().properties(text("name")).create();
  assert(new S.OfRelation.Builder().links("a", "b").properties((p) => p.name("pins").of(list_of(Pin))).create().validate().length === 0);
  // Builders finalize as every builder does.
  const copied = new S.OfIndexed.Builder(Tags).clone();
  assert(copied !== Tags && copied.item === Tags.item);
  assert(Tags.equals(Tags) && !copied.equals(Tags)); // schema data are equal only to themselves
  assert(new S.OfIndexed.Builder(Tags).of(Int).update() === Tags && (Tags.item as S.OfNative.Data).type === BigInt);
  raises(ValueError, () => new S.OfIndexed.Builder(Tags).create(), "create() is only valid without a source instance");
}

## LST-02 · Proxies read a list as a read-only sequence and set it from a sequence of items

In [ ]:
{
  let p = B.Part().name("p").tags(["a", "b"]).grid([[1n, 2n], [3n]]).shapes(
    [(u: any) => u.circle(1.5), (u: any) => u.label("x")]).ports([(o: any) => o.name("in")]).create();
  assert(same(p.tags, ["a", "b"]) && Array.isArray(p.tags) && Object.isFrozen(p.tags) && equal(p.grid, [[1n, 2n], [3n]]));
  assert(p.shapes[0].circle === 1.5 && p.shapes[1].label === "x" && p.ports[0].name === "in");
  assert(p.ports[0].owner() === p && p.shapes[1].owner() === p);
  assert(same(B.member(p, "tags") as unknown[], ["a", "b"]));
  p = B.Part(p).tags(["c"]).update();
  assert(same(p.tags, ["c"]));
  p = B.Part(p).tags([]).update();
  assert(same(p.tags, [])); // an empty list is a value, unlike an absent one
  p = B.Part(p).clear("tags").update();
  assert(p.tags === null);
  raises(AttributeError, () => { (p as any).tags = []; }, "read-only");
  // An item written with no value is left out: a union value without a branch is no value.
  assert(B.Part().shapes([(u: any) => u, (u: any) => u.label("y")]).create().shapes.length === 1);
  // Items are written under the item schema, and a list takes a list or a Spec.
  raises(TypeError, () => B.Part().tags("ab"), "property 'tags' takes a list or a Spec, got str");
  raises(TypeError, () => B.Part().ports(["x"]), "property 'ports' takes a value object or a Spec, got str");
  raises(TypeError, () => B.Part().property("name", (q: any) => q.value((a: any) => a.as_indexed(() => null))),
    "property 'name' does not hold a list");
  raises(TypeError, () => B.Part().property("tags", (q: any) => q.value((a: any) => a.as_object(() => null))),
    "property 'tags' does not hold an object");
}

## LST-03 · A Spec edits a list through its builder, keeping the identities of its value objects

In [ ]:
{
  let p = B.Part().ports([(o: any) => o.name("a"), (o: any) => o.name("b")]).create();
  const [a, b] = p.ports;
  p = B.Part(p).ports((items: any) => items.item(0, (v: any) => v.as_object((o: any) => o.name("a2"))).append(
    (v: any) => v.as_object((o: any) => o.name("c")))).update();
  assert(p.ports[0] === a && a.name === "a2" && p.ports[1] === b && p.ports[2].name === "c" && p.ports[2].owner() === p);
  const names: boolean[] = [];
  B.Part(p).ports((items: any) => items.items((v: any) => v.as_object((o: any) => names.push(o.has("name")))));
  assert(same(names, [true, true, true]));
  for (const bad of [3, -1]) {
    raises(LookupError, () => B.Part(p).ports((items: any) => items.item(bad, () => null)), `the list has no item ${bad}`);
    raises(LookupError, () => B.Part(p).ports((items: any) => items.remove(bad)), `the list has no item ${bad}`);
  }
  // Value objects in a list link each other, and to reference objects.
  const hub = B.Hub().name("hub").create();
  p = B.Part(p).ports((items: any) => items.item(0, (v: any) => v.as_object((o: any) => o.wires((x: any) => x.sink(b)).wires(
    (x: any) => x.sink(hub))))).update();
  assert(p.ports[0] === a);
  assert(equal(s0(Plain.ToPlain(store)(Part, p)).get("ports").slice(0, 2), map([
    { $id: "s1", name: "a2", wires: [{ sink: { $ref: "s2" } }, { sink: { $ref: "s3", $schema: "Hub" } }] },
    { $id: "s2", name: "b", fed: [{ source: { $ref: "s1" } }] }])));
  // Removing an item removes it and the entries linking it; the others keep their identities.
  p = B.Part(p).ports((items: any) => items.remove(1)).update();
  assert(p.ports.length === 2 && p.ports[0] === a && same(p.ports.map((x: any) => x.name), ["a2", "c"]));
  assert(equal(s0(Plain.ToPlain(store)(Part, p)).get("ports")[0].get("wires"), map([{ sink: { $ref: "s1", $schema: "Hub" } }])));
  // Lists of lists keep identities too, by position; a list appended is new.
  p = B.Part().banks([[(o: any) => o.name("x")]]).create();
  const x = p.banks[0][0];
  p = B.Part(p).banks((rows: any) => rows.item(0, (v: any) => v.as_indexed((row: any) => row.item(
    0, (w: any) => w.as_object((o: any) => o.name("x2"))))).append((v: any) => v.as_indexed(
    (row: any) => row.append((w: any) => w.as_object((o: any) => o.name("y")))))).update();
  assert(p.banks[0][0] === x && x.name === "x2" && p.banks[1][0].name === "y" && p.banks[1][0].owner() === p);
  // Clearing the list removes every item.
  p = B.Part(p).banks((rows: any) => rows.clear()).update();
  assert(same(p.banks, []));
}

## LST-04 · Value objects in a list are copied into another owner and by a clone, and removed with their owner's list

In [ ]:
{
  const hub = B.Hub().name("hub").create();
  let p = B.Part().name("p").ports([(o: any) => o.name("a"), (o: any) => o.name("b")]).create();
  const [a, b] = p.ports;
  p = B.Part(p).ports((items: any) => items.item(0, (v: any) => v.as_object((o: any) => o.wires((x: any) => x.sink(b)).wires(
    (x: any) => x.sink(hub))))).update();
  const before = Plain.ToPlain(store)(Part, p);
  // A clone copies every list, and the entries among the copies link the copies.
  const clone = B.Part(p).clone();
  assert(clone.ports[0] !== a && clone.ports[1] !== b && clone.ports[0].owner() === clone);
  assert(equal(Plain.ToPlain(store)(Part, clone), before));
  assert(equal(Plain.ToPlain(store)(Part, p), before)); // the source is untouched
  // Setting a value object into another owner's list copies it, with its entries.
  const q = B.Part().name("q").ports([a]).create();
  assert(q.ports[0] !== a && q.ports[0].owner() === q && q.ports[0].name === "a");
  assert(equal(s0(Plain.ToPlain(store)(Part, q)).get("ports"), map([
    { name: "a", wires: [{ sink: { $ref: "s1" } }, { sink: { $ref: "s2", $schema: "Hub" } }] }])));
  // Replacing the list removes the value objects no longer in it, with their entries.
  p = B.Part(p).ports([(o: any) => o.name("c")]).update();
  assert(equal(s0(Plain.ToPlain(store)(Hub, hub)).get("feeds"), map([{ source: { $ref: "s1" } }, { source: { $ref: "s2" } }])));
  assert(same(Reachable.of(hub), [hub, clone, q])); // p's ports are gone, so nothing links p
  p = B.Part(clone).clear("ports").update();
  assert(same(Reachable.of(hub), [hub, q]));
  // A value object removed takes the value objects it holds with it, and their entries.
  p = B.Part().racks([(r: any) => r.name("r").port((o: any) => o.name("p").wires((x: any) => x.sink(hub)))]).create();
  assert(p.racks[0].port.owner() === p.racks[0] && same(Reachable.of(hub), [hub, q, p]));
  p = B.Part(p).racks((items: any) => items.remove(0)).update();
  assert(same(Reachable.of(hub), [hub, q]));
}

## LST-05 · Snapshots write a list as an array, its value objects nested with `$id`, and decode it strictly

In [ ]:
{
  const hub = B.Hub().name("hub").create();
  let p = B.Part().name("p").tags(["a", "b"]).grid([[1n, 2n], []]).shapes([(u: any) => u.label("x")]).banks(
    [[], [(o: any) => o.name("x"), (o: any) => o.name("y")]]).create();
  const [, y] = p.banks[1];
  p = B.Part(p).banks((rows: any) => rows.item(1, (v: any) => v.as_indexed((row: any) => row.item(
    0, (w: any) => w.as_object((o: any) => o.wires((e: any) => e.sink(y)).wires((e: any) => e.sink(hub))))))).update();
  const text = JSON.ToJSON(store).Reachable(Part, p);
  assert(text === (
    '{"root": "s0", "objects": {"s0": {"name": "p", "tags": ["a", "b"], "grid": [[1, 2], []], "banks": [[], ' +
    '[{"$id": "s2", "name": "x", "wires": [{"sink": {"$ref": "s3"}}, {"sink": {"$ref": "s1", "$schema": "Hub"}}]}, ' +
    '{"$id": "s3", "name": "y", "fed": [{"source": {"$ref": "s2"}}]}]], "shapes": [{"label": "x"}]}, ' +
    '"s1": {"name": "hub", "feeds": [{"source": {"$ref": "s2"}}]}}}'));
  const back = JSON.FromJSON(B).Reachable(Part, text) as any;
  assert(JSON.ToJSON(store).Reachable(Part, back) === text);
  assert(back.banks[1][0].owner() === back && equal(back.grid, [[1n, 2n], []]));
  assert(YAML.ToYAML(store).Reachable(Part, YAML.FromYAML(B).Reachable(Part, YAML.ToYAML(store).Reachable(Part, p))) ===
    YAML.ToYAML(store).Reachable(Part, p));
  assert(new C.OfObject(Part, back).compare(new C.OfObject(Part, p)) === null); // its port links another hub, by identity
  // Decoding is strict, with paths into the lists.
  decode_error(snapshot({ tags: "a" }), "$.objects.s0.tags: a list must be an array, got str");
  decode_error(snapshot({ tags: ["a", 1n] }), "$.objects.s0.tags[1]: expected str, got int");
  decode_error(snapshot({ grid: [[1n], "x"] }), "$.objects.s0.grid[1]: a list must be an array, got str");
  decode_error(snapshot({ ports: [{ nope: 1n }] }), "$.objects.s0.ports[0].nope: the value object has no property 'nope'");
  decode_error(snapshot({ ports: [{ $id: "s1" }, { $id: "s1" }] }), "$.objects.s0.ports[1].$id: 's1' is the symbol of two objects");
  decode_error(snapshot({ ports: [{ wires: [{ sink: { $ref: "s9" } }] }] }),
    "$.objects.s0.ports[0].wires[0].sink: unresolved reference 's9': no value object in the snapshot has this symbol");
  decode_error(snapshot({ ports: [{ wires: [{ sink: { $ref: 1n } }] }] }),
    "$.objects.s0.ports[0].wires[0].sink: a reference is {'$ref': symbol, '$schema': name}");
  decode_error(snapshot({ ports: [{ wires: [1n] }] }), "$.objects.s0.ports[0].wires[0]: an entry must be a mapping, got int");
  decode_error(snapshot({ shapes: [{}] }), "$.objects.s0.shapes[0]: a union value holds exactly one branch, got 0");
}

## LST-06 · Validation checks each item against the item schema, at its index

In [ ]:
{
  const validate = Validators.Validate(B);
  let p = B.Part().name("p").tags(["a"]).ports([(o: any) => o.name("a"), (o: any) => o.name("b")]).create();
  p = B.Part(p).ports((items: any) => items.item(0, (v: any) => v.as_object((o: any) => o.wires(
    (x: any) => x.sink(p.ports[1]))))).update();
  assert(validate(Part, p).length === 0 && validate.Reachable(Part, p).length === 0);

  /** A Part whose property `name` holds what `write(OfAny)` writes. */
  const written = (name: string, write: (a: any) => void) => new Written((v) => v.property(name, (q: any) => q.value(write)));
  const items = (...writes: ((a: any) => void)[]) => (a: any) => a.as_indexed((list: any) => {
    for (const w of writes) list.append(w);
  });
  const native = (value: unknown) => (a: any) => a.as_native((n: any) => n.set(value));
  const port = (value: unknown) => (a: any) => a.as_object((o: any) => o.property("name", (q: any) => q.value(native(value))));
  assert(same(validate(Part, written("tags", items(native("a"), native(1n)))), ["Part#0.tags[1]: expected str, got int"]));
  assert(same(validate(Part, written("tags", native("a"))), ["Part#0.tags: expected a list, got str"]));
  assert(same(validate(Part, written("name", items())), ["Part#0.name: expected str, got a list"]));
  assert(same(validate(Part, written("ports", items(native("x")))), ["Part#0.ports[0]: expected a value object, got str"]));
  assert(same(validate(Part, written("ports", items(port("a"), port(2n)))), ["Part#0.ports[1].name: expected str, got int"]));
  assert(same(validate(Part, written("grid", items(items(native(1n)), items(native("2"))))),
    ["Part#0.grid[1][0]: expected int, got str"]));
  // `properties_of` gives a list as an object whose `values` are its items, read the same way.
  const values = Validators.properties_of(p) as Map<string, any>;
  assert(same(values.get("tags").values, ["a"]) &&
    same(values.get("ports").values.map((o: any) => Validators.properties_of(o).get("name")), ["a", "b"]));
}

## LST-07 · Lists compare item by item: lexicographically for ordered natives, and a prefix is less

In [ ]:
{
  const Tags = Part.properties.get("tags")?.type as S.OfIndexed.Data;
  const tags = (values: unknown[], schema: S.OfIndexed.Data = Tags) => {
    const recorded = new C.OfIndexed(schema);
    for (const value of values) recorded.append((a: any) => a.as_native((n: any) => n.set(value)));
    return recorded;
  };
  assert(tags(["a", "b"]).compare(tags(["a", "b"])) === 0 && tags([]).compare(tags([])) === 0);
  assert(tags(["a"]).compare(tags(["b"])) === -1 && tags(["b"]).compare(tags(["a", "z"])) === 1);
  assert(tags(["a"]).compare(tags(["a", "b"])) === -1 && tags(["a", "b"]).compare(tags(["a"])) === 1);
  const Flags = new S.OfIndexed.Builder().of((t) => t.as_native(Boolean)).create();
  assert(tags([true], Flags).compare(tags([false], Flags)) === null); // booleans are not ordered
  assert(tags([true], Flags).compare(tags([true], Flags)) === 0);
  assert(tags(["a"]).compare(tags([true], Flags)) === null); // different schemas
  // Objects compare their lists deeply, value objects included, and links among them by path.
  let p = B.Part().tags(["a"]).ports([(o: any) => o.name("a"), (o: any) => o.name("b")]).create();
  p = B.Part(p).ports((items: any) => items.item(0, (v: any) => v.as_object((o: any) => o.wires(
    (x: any) => x.sink(p.ports[1]))))).update();
  const clone = B.Part(p).clone();
  assert(new C.OfObject(Part, p).compare(new C.OfObject(Part, clone)) === 0);
  let crossed = B.Part(p).clone();
  crossed = B.Part(crossed).ports((items: any) => items.item(0, (v: any) => v.as_object((o: any) => o.wires(
    (x: any) => x.sink(crossed.ports[0]))))).update(); // a self-loop too: not the same links
  assert(new C.OfObject(Part, p).compare(new C.OfObject(Part, crossed)) === null);
  assert(new C.OfObject(Part, B.Part().tags([]).create()).compare(new C.OfObject(Part, B.Part().create())) === null);
  assert(new C.OfObject(Part, B.Part().tags(["a"]).create()).compare(new C.OfObject(Part, B.Part().tags(["b"]).create())) === null);
}

## LST-08 · Reachability follows the value objects in lists, and entry properties hold lists

In [ ]:
{
  const hub = B.Hub().name("hub").create();
  let p = B.Part().banks([[(o: any) => o.name("x")]]).create();
  p = B.Part(p).banks((rows: any) => rows.item(0, (v: any) => v.as_indexed((row: any) => row.item(
    0, (w: any) => w.as_object((o: any) => o.wires((e: any) => e.sink(hub))))))).update();
  assert(same(Reachable.of(p), [p, hub]) && same(Reachable.of(hub), [hub, p])); // a value object reached brings in its owner
  assert(same([...Reachable.targets([p])], [hub.identity()]));
  // Entries key and compare by their lists, in order: an equal entry is elided.
  p = B.Part().labelled((x: any) => x.hub(hub).labels(["a", "b"])).labelled((x: any) => x.hub(hub).labels(["a", "b"])).labelled(
    (x: any) => x.hub(hub).labels(["b", "a"])).create();
  assert(equal(s0(Plain.ToPlain(store)(Part, p)).get("labelled"), map([
    { hub: { $ref: "s1", $schema: "Hub" }, labels: ["a", "b"] },
    { hub: { $ref: "s1", $schema: "Hub" }, labels: ["b", "a"] }])));
  const back = Plain.FromPlain(B).Reachable(Part, Plain.ToPlain(store).Reachable(Part, p));
  assert(equal(Plain.ToPlain(store).Reachable(Part, back as any), Plain.ToPlain(store).Reachable(Part, p)));
  assert(Validators.Validate(B).Reachable(Part, p).length === 0);
}

## LST-09 · The list classes behave through their protocols

In [ ]:
{
  const seen: unknown[] = [];
  // Plain's writer: items read back, edited, removed and cleared.
  const out = new Map<string, any>();
  const writer = new Plain._AnyWriter(out, "tags", Part.properties.get("tags")?.type as S.OfAny.Data, () => new Map(), () => null);
  writer.as_indexed((items) => items.append((a) => a.as_native((n) => n.set("a"))).append(
    (a) => a.as_native((n) => n.set("b"))));
  writer.as_indexed((items) => items.items((a) => a.as_native((n) => seen.push(n.get()))));
  writer.as_indexed((items) => items.item(1, (a) => a.as_native((n) => n.set("c"))).remove(0));
  assert(same(seen, ["a", "b"]) && equal(out, map({ tags: ["c"] })));
  writer.as_indexed((items) => items.item(0, (a) => a.as_native((n) => n.clear()))); // an item cleared is removed
  writer.as_indexed((items) => items.append((a) => a.as_native((n) => n.set("d"))).clear());
  assert(equal(out, map({ tags: [] })));
  raises(TypeError, () => new Plain._AnyWriter(out, "name", Part.properties.get("name")?.type as S.OfAny.Data, () => new Map(),
    () => null).as_indexed(() => null), "property 'name' is not a list");
  raises(LookupError, () => writer.as_indexed((items) => items.remove(0)), "the list has no item 0");
  // The validation recorder keeps what it is given, and reads it back.
  const record = new Validators._ObjectRecord();
  record.property("tags", (p) => p.value((a) => a.as_indexed((items) => items.append(
    (v) => v.as_native((n) => n.set("a"))).append((v) => v.as_native((n) => n.set("b"))))));
  seen.length = 0;
  record.property("tags", (p) => p.value((a) => a.as_indexed((items) => items.items(
    (v) => v.as_native((n) => seen.push(n.get()))).item(0, (v) => v.as_native((n) => n.set("z"))).remove(1))));
  assert(same(seen, ["a", "b"]) && same((record.values.get("tags") as Validators._ListRecord).values, ["z"]));
  const replayed = new Validators._ObjectRecord();
  record.accept(replayed);
  assert(same((replayed.values.get("tags") as Validators._ListRecord).values, ["z"]));
  record.property("tags", (p) => p.value((a) => a.as_indexed((items) => items.clear())));
  assert(same((record.values.get("tags") as Validators._ListRecord).values, []));
  raises(LookupError, () => record.property("tags", (p) => p.value((a) => a.as_indexed((items) => items.remove(2)))),
    "the list has no item 2");
  // The comparison recorder too.
  const recorded = new C.OfAny(Part.properties.get("tags")?.type as S.OfAny.Data);
  recorded.as_indexed((items) => items.append((v) => v.as_native((n) => n.set("a"))).append(
    (v) => v.as_native((n) => n.set("b"))));
  seen.length = 0;
  recorded.as_indexed((items) => items.items((v) => v.as_native((n) => seen.push(n.get()))).item(
    0, (v) => v.as_native((n) => n.set("z"))).remove(1));
  assert(same(seen, ["a", "b"]));
  const other = new C.OfAny(Part.properties.get("tags")?.type as S.OfAny.Data);
  other.as_indexed((items) => items.append((v) => v.as_native((n) => n.set("z"))).append(
    (v) => v.as_native((n) => n.clear()))); // an item written with no value is left out
  assert(recorded.compare(other) === 0);
  recorded.as_indexed((items) => items.clear());
  assert(recorded.compare(other) === -1);
  const named = new C.OfAny(Part.properties.get("name")?.type as S.OfAny.Data);
  named.as_native((n) => n.set("z"));
  assert(recorded.compare(named) === null && named.compare(recorded) === null); // a list and a native are incomparable
  raises(LookupError, () => recorded.as_indexed((items) => items.remove(5)), "the list has no item 5");
  raises(TypeError, () => new C.OfAny(Part.properties.get("name")?.type as S.OfAny.Data).as_indexed(() => null),
    "the schema is not a list schema");
  // The reachability collector discards lists, keeping no items, and follows the value objects appended.
  const nested: any[] = [];
  const collector = new Reachable._Collector((t) => seen.push(t));
  collector.property("ports", (p) => p.value((a) => a.as_indexed((items) => nested.push(items))));
  const discarded = nested[0];
  assert(discarded.items((v: unknown) => seen.push(v)) === discarded && discarded.clear() === discarded);
  discarded.append((v: any) => v.as_object((o: any) => nested.push(o)));
  assert(nested[1] instanceof Reachable._Collector);
  for (const method of [() => discarded.item(0, (v: unknown) => seen.push(v)), () => discarded.remove(0)]) {
    raises(LookupError, method, "the list has no item 0");
  }
  // The proxy builder: removing an item, and clearing.
  let p = B.Part().tags(["a", "b", "c"]).create();
  p = B.Part(p).tags((items: any) => items.remove(1)).update();
  assert(same(p.tags, ["a", "c"]));
}

## LST-10 · A key schema makes a list keyed, and an extent bounds a positional one

In [ ]:
{
  const Attrs = new S.OfIndexed.Builder().key(Str).of(Float).create();
  assert(!Attrs.positional && Attrs.validate().length === 0 && Attrs.minimum === 0n && Attrs.capacity === null);
  const Row = new S.OfIndexed.Builder().key(Int).of(Str).extent({ minimum: 1n, maximum: 3n }).create(); // an int key: positional
  assert(Row.positional && Row.minimum === 1n && Row.capacity === 3n && Row.validate().length === 0);
  assert(new S.OfIndexed.Builder().of(Str).extent({ maximum: 4n }).create().extent?.equals(new S.OfIndexed.Extent(0n, 4n)));
  assert(!new S.OfIndexed.Extent(0n, 4n).equals(new S.OfIndexed.Extent(1n, 4n)));
  assert((Part.properties.get("cells")?.type as S.OfIndexed.Data).key === Cell && Part.validate().length === 0 && Labels.validate().length === 0);
  // A key is a value: never a reference object, and its value objects have no adjacencies.
  const Card = new S.OfObject.Builder().ref().create();
  assert(same(new S.OfIndexed.Builder().key(Card).of(Str).create().validate(), ["key: a reference object schema cannot be a key"]));
  assert(same(new S.OfIndexed.Builder().key(Port).of(Str).create().validate(), ["key: a value object in a key cannot have adjacencies"]));
  assert(same(new S.OfIndexed.Builder().key(list_of(Port)).of(Str).create().validate(),
    ["key: a value object in a key cannot have adjacencies"]));
  assert(same(new S.OfIndexed.Builder().key((t) => t.as_native((n) => n.type(BigInt).bits(0n))).of(Str).create().validate(),
    ["key: a width in bits must be a positive int or a term, got 0"]));
  // An extent is for positional lists, of ints, its minimum at most its maximum.
  assert(same(new S.OfIndexed.Builder().key(Str).of(Str).extent({ minimum: 0n, maximum: 1n }).create().validate(),
    ["an extent bounds a positional list, whose keys are ints"]));
  assert(same(new S.OfIndexed.Builder().of(Str).extent({ minimum: 5n, maximum: 2n }).create().validate(),
    ["an extent's minimum 5 exceeds its maximum 2"]));
  const Odd = new S.OfIndexed.Data({ item: new S.OfNative.Data(String), extent: new S.OfIndexed.Extent("1" as never, null) });
  assert(same(Odd.validate(), ["an extent's minimum and maximum are ints or terms, got '1' and None"]));
  assert(Odd.minimum === 0n && Odd.capacity === null); // an extent that is not ints bounds nothing
}

## LST-11 · Proxies read a keyed list as a read-only mapping, in insertion order, set from a mapping or pairs

In [ ]:
{
  let p = B.Part().attrs(new Map([["a", 1.5], ["b", -0.0]])).cells([[(c: any) => c.r(1n).c(2n), 7n], [(c: any) => c.r(0n).c(0n), 9n]]).create();
  assert(p.attrs instanceof Proxies.OfIndexed.Map && same(p.attrs.keys(), ["a", "b"]) && Object.is(p.attrs.get("b"), -0));
  assert(p.attrs.get("a") === 1.5 && p.attrs.get("z") === undefined && p.attrs.has("b") && p.attrs.size === 2);
  assert(equal([...p.attrs], [["a", 1.5], ["b", -0.0]]) && same(p.attrs.values(), [1.5, -0.0]));
  raises(TypeError, () => { (p.attrs as any).c = 1.0; });
  const first = p.cells.keys()[0];
  assert(first.r === 1n && first.owner() === p && p.cells.get(first) === 7n);
  const other = B.Part().cells([[(c: any) => c.r(1n).c(2n), 1n]]).create().cells.keys()[0];
  assert(p.cells.get(other) === 7n); // a value object key matches by structure
  // `put` replaces a value in place, keeping the order; a new key goes last; the value of a value object keeps its identity.
  p = B.Part(p).attrs((m: any) => m.put((k: any) => k.as_native((n: any) => n.set("a")), (v: any) => v.as_native((n: any) => n.set(2.0)))
    .put((k: any) => k.as_native((n: any) => n.set("c")), (v: any) => v.as_native((n: any) => n.set(3.0)))).update();
  assert(equal(p.attrs.entries(), [["a", 2.0], ["b", -0.0], ["c", 3.0]]));
  const X = new Uint8Array([120]);
  p = B.Part(p).named(new Map([[X, (o: any) => o.name("x")]])).update();
  const x = p.named.get(new Uint8Array([120]));
  p = B.Part(p).named((m: any) => m.put((k: any) => k.as_native((n: any) => n.set(new Uint8Array([120]))),
    (v: any) => v.as_object((o: any) => o.name("x2")))).update();
  assert(p.named.get(X) === x && x.name === "x2" && x.owner() === p);
  assert(equal(B.Part().attrs(p.attrs).create().attrs.entries(), p.attrs.entries())); // a Map is a mapping
  assert(B.Part().attrs(new Map()).create().attrs.size === 0); // empty, unlike absent
  // Through the builder: by key and by position.
  const seen: unknown[] = [];
  B.Part(p).attrs((m: any) => m.pairs((item: any) => item.key((k: any) => k.as_native((n: any) => seen.push(n.get())))
    .value((v: any) => v.as_native((n: any) => seen.push(n.get())))));
  assert(same(seen, ["a", 2.0, "b", -0.0, "c", 3.0]));
  seen.length = 0;
  B.Part(p).attrs((m: any) => m.at((k: any) => k.as_native((n: any) => n.set("b")), (v: any) => v.as_native((n: any) => seen.push(n.get())))
    .items((v: any) => v.as_native((n: any) => seen.push(n.get()))));
  assert(same(seen, [-0.0, 2.0, -0.0, 3.0]));
  p = B.Part(p).attrs((m: any) => m.discard((k: any) => k.as_native((n: any) => n.set("b"))).remove(0)).update();
  assert(equal(p.attrs.entries(), [["c", 3.0]]));
  p = B.Part(p).attrs((m: any) => m.item(0, (v: any) => v.as_native((n: any) => n.clear())) // an item with no value is left out
    .put((k: any) => k.as_native((n: any) => n.set("d")), (v: any) => v.as_native((n: any) => n.clear()))).update();
  assert(p.attrs.size === 0);
  p = B.Part(p).attrs(new Map([["a", 1.0]])).update();
  p = B.Part(p).attrs((m: any) => m.clear()).update();
  assert(p.attrs.size === 0);
  const keyA = (k: any) => k.as_native((n: any) => n.set("a"));
  for (const [wrong, error, message] of [
    [(m: any) => m.append(() => null), TypeError, "a keyed list takes put, not append"],
    [(m: any) => m.put(() => null, () => null), ValueError, "a key needs a value"],
    [(m: any) => m.at(keyA, () => null), LookupError, "the list has no item with this key"],
    [(m: any) => m.discard(keyA), LookupError, "the list has no item with this key"],
  ] as [(m: any) => unknown, Function, string][]) {
    raises(error, () => B.Part(p).attrs(wrong), message);
  }
  raises(TypeError, () => B.Part().attrs("x"), "property 'attrs' takes a mapping, (key, value) pairs or a Spec, got str");
  raises(TypeError, () => B.Part().attrs([["a"]]), "property 'attrs' takes (key, value) pairs, got list");
  raises(TypeError, () => B.Part().tags(new Map([["a", "b"]])), "property 'tags' takes a list or a Spec, got dict");
}

## LST-12 · A positional list's keys are its positions, from its extent's minimum

In [ ]:
{
  let p = B.Part().row([10n, 20n]).create();
  const keys: unknown[] = [];
  B.Part(p).row((r: any) => r.pairs((item: any) => item.key((k: any) => k.as_native((n: any) => keys.push(n.get())))));
  assert(same(keys, [1n, 2n]));
  const at = (key: unknown) => (k: any) => k.as_native((n: any) => n.set(key));
  const seen: unknown[] = [];
  B.Part(p).row((r: any) => r.at(at(2n), (v: any) => v.as_native((n: any) => seen.push(n.get()))));
  assert(same(seen, [20n]));
  p = B.Part(p).row((r: any) => r.put(at(3n), (v: any) => v.as_native((n: any) => n.set(30n))) // the next key appends
    .put(at(1n), (v: any) => v.as_native((n: any) => n.set(11n)))).update(); // a key held replaces
  assert(same(p.row, [11n, 20n, 30n]));
  p = B.Part(p).row((r: any) => r.discard(at(2n))).update();
  assert(same(p.row, [11n, 30n]));
  for (const [wrong, error, message] of [
    [(r: any) => r.put(at(5n), () => null), LookupError, "the list has no item 5"],
    [(r: any) => r.at(at(0n), () => null), LookupError, "the list has no item 0"],
    [(r: any) => r.discard(at(3n)), LookupError, "the list has no item 3"],
    [(r: any) => r.at(at("1"), () => null), TypeError, "a positional list's keys are ints, got str"],
    [(r: any) => r.at(() => null, () => null), ValueError, "a key needs a value"],
  ] as [(r: any) => unknown, Function, string][]) {
    raises(error, () => B.Part(p).row(wrong), message);
  }
  // Validation labels a positional list's items by key, and checks its extent.
  const validate = Validators.Validate(B);
  assert(validate(Part, B.Part().row([1n, 2n, 3n]).create()).length === 0);
  assert(same(validate(Part, B.Part().row([1n, 2n, 3n, 4n]).create()), ["Part#0.row: 4 items are more than the extent 1..3 holds"]));
  assert(same(validate(Part, new Written((v) => v.property("row", (q: any) => q.value((a: any) => a.as_indexed(
    (r: any) => r.append((i: any) => i.as_native((n: any) => n.set(1n))).append((i: any) => i.as_native((n: any) => n.set("2")))))))),
  ["Part#0.row[2]: expected int, got str"]));
}

## LST-13 · A keyed list is a mapping from its keys' text when that never starts with `$`, else an array of keys and values

In [ ]:
{
  const hub = B.Hub().name("hub").create();
  let p = B.Part().name("p").attrs(new Map([["a", 1.5], ["$ref", 2.0]])) // str keys may be anything, so they are written as items
    .weights([[1.5, "x"], [NaN, "n"], [-0.0, "z"]]) // pairs keep -0.0
    .flags(new Map([[true, "on"], [false, "off"]]))
    .named(new Map([[new Uint8Array([0]), (o: any) => o.name("zero")], [new Uint8Array([120]), (o: any) => o.name("x")]]))
    .cells([[(c: any) => c.r(1n).c(2n).w(0.5).path([1n]).meta(new Map([["k", "v"]])), 7n]]).create();
  p = B.Part(p).named((m: any) => m.put((k: any) => k.as_native((n: any) => n.set(new Uint8Array([120]))),
    (v: any) => v.as_object((o: any) => o.wires((e: any) => e.sink(hub))))).update();
  const text = JSON.ToJSON(store).Reachable(Part, p);
  assert(text === (
    '{"root": "s0", "objects": {"s0": {"name": "p", "attrs": [{"key": "a", "value": 1.5}, {"key": "$ref", "value": 2.0}], ' +
    '"weights": {"1.5": "x", "NaN": "n", "-0.0": "z"}, ' +
    '"flags": {"true": "on", "false": "off"}, "named": {"AA==": {"name": "zero"}, ' +
    '"eA==": {"$id": "s2", "name": "x", "wires": [{"sink": {"$ref": "s1", "$schema": "Hub"}}]}}, ' +
    '"cells": [{"key": {"r": 1, "c": 2, "w": 0.5, "path": [1], "meta": [{"key": "k", "value": "v"}]}, "value": 7}]}, ' +
    '"s1": {"name": "hub", "feeds": [{"source": {"$ref": "s2"}}]}}}'));
  const back = JSON.FromJSON(B).Reachable(Part, text) as any;
  assert(JSON.ToJSON(store).Reachable(Part, back) === text && back.named.get(new Uint8Array([120])).owner() === back);
  assert(YAML.ToYAML(store).Reachable(Part, YAML.FromYAML(B).Reachable(Part, YAML.ToYAML(store).Reachable(Part, p)) as any) ===
    YAML.ToYAML(store).Reachable(Part, p));
  // Decoding is strict: only a key's canonical text, mappings and arrays where they belong, each key once.
  decode_error(snapshot({ weights: [] }), "$.objects.s0.weights: a keyed list must be a mapping, got list");
  decode_error(snapshot({ weights: { "1.50": "x" } }), "$.objects.s0.weights[\"1.50\"]: expected the text of a float key, got '1.50'");
  decode_error(snapshot({ weights: { one: "x" } }), "$.objects.s0.weights.one: expected the text of a float key, got 'one'");
  decode_error(snapshot({ flags: { yes: "x" } }), "$.objects.s0.flags.yes: expected the text of a bool key, got 'yes'");
  decode_error(snapshot({ named: { "!!": {} } }), "$.objects.s0.named[\"!!\"]: expected the text of a bytes key, got '!!'");
  decode_error(snapshot({ weights: { "1.5": true } }), '$.objects.s0.weights["1.5"]: expected str, got bool');
  decode_error(snapshot({ attrs: { a: 1.5 } }), "$.objects.s0.attrs: a keyed list must be an array, got dict");
  decode_error(snapshot({ cells: {} }), "$.objects.s0.cells: a keyed list must be an array, got dict");
  decode_error(snapshot({ cells: [{ key: { r: 1n } }] }),
    "$.objects.s0.cells[0]: an item of a keyed list is {'key': key, 'value': value}");
  decode_error(snapshot({ cells: [{ key: { r: "1" }, value: 1n }] }), "$.objects.s0.cells[0].key.r: expected int, got str");
  decode_error(snapshot({ cells: [{ key: { $id: "s5" }, value: 1n }] }),
    "$.objects.s0.cells[0].key.$id: the value object has no property '$id'");
  const sameKey = { r: 1n, w: 0.5, path: [1n], meta: [{ key: "k", value: "v" }] };
  decode_error(snapshot({ cells: [{ key: sameKey, value: 1n }, { key: sameKey, value: 2n }] }),
    "$.objects.s0.cells[1].key: the same key as item 0");
}

## LST-14 · Keyed lists are validated by position, compared as mappings, and followed for reachability

In [ ]:
{
  const validate = Validators.Validate(B);
  let p = B.Part().attrs(new Map([["a", 1.0], ["b", 2.0]])).cells([[(c: any) => c.r(1n).meta(new Map([["k", "v"]])), 1n]]).create();
  assert(validate(Part, p).length === 0);

  /** A Part whose keyed list `name` holds `items`: [key, value] writers, or a lone value writer appended. */
  const written = (name: string, ...items: unknown[]) => new Written((v) => v.property(name, (q: any) => q.value((a: any) =>
    a.as_indexed((list: any) => {
      for (const item of items) {
        if (Array.isArray(item)) list.put(item[0], item[1]);
        else list.append(item);
      }
    }))));
  const native = (value: unknown) => (a: any) => a.as_native((n: any) => n.set(value));
  const cell = (meta: string) => (a: any) => a.as_object((o: any) => o.property("meta", (q: any) => q.value(
    (m: any) => m.as_indexed((list: any) => list.put(native("k"), native(meta))))));
  assert(same(validate(Part, written("attrs", [native("a"), native(1.0)], [native("a"), native(2.0)])),
    ["Part#0.attrs[1].key: the same key as item 0"]));
  assert(same(validate(Part, written("cells", [cell("v"), native(1n)], [cell("v"), native(2n)], [cell("w"), native(3n)])),
    ["Part#0.cells[1].key: the same key as item 0"]));
  assert(same(validate(Part, written("attrs", [native(1n), native("x")])),
    ["Part#0.attrs[0].key: expected str, got int", "Part#0.attrs[0]: expected float, got str"]));
  assert(same(validate(Part, written("attrs", native(1.0))), ["Part#0.attrs[0].key: not set"]));
  const values = Validators.properties_of(p).get("attrs") as Validators._ListRecord;
  assert(same(values.keys, ["a", "b"]) && same(values.values, [1.0, 2.0]));
  const replayed = new Validators._ObjectRecord();
  const record = new Validators._ObjectRecord();
  p.accept(record);
  record.accept(replayed); // a recorded keyed list replays its keys
  assert(same((replayed.values.get("attrs") as Validators._ListRecord).keys, ["a", "b"]));
  // Keyed lists compare as mappings: the same keys with equal values, in any order.
  const compare = (a: Map<string, number>, b: Map<string, number>) =>
    new C.OfObject(Part, B.Part().attrs(a).create()).compare(new C.OfObject(Part, B.Part().attrs(b).create()));
  assert(compare(new Map([["a", 1.0], ["b", 2.0]]), new Map([["b", 2.0], ["a", 1.0]])) === 0);
  assert(compare(new Map([["a", 1.0]]), new Map([["a", 2.0]])) === null &&
    compare(new Map([["a", 1.0]]), new Map([["a", 1.0], ["b", 2.0]])) === null);
  assert(compare(new Map([["a", 1.0]]), new Map([["b", 1.0]])) === null);
  // Entries key by keyed lists as mappings too, and reachability follows the value objects they hold.
  const hub = B.Hub().name("hub").create();
  p = B.Part().labelled((x: any) => x.hub(hub).notes(new Map([["a", "1"], ["b", "2"]]))).labelled(
    (x: any) => x.hub(hub).notes(new Map([["b", "2"], ["a", "1"]]))).named(new Map([[new Uint8Array([120]), (o: any) => o.name("x")]])).create();
  assert((s0(Plain.ToPlain(store)(Part, p)).get("labelled") as unknown[]).length === 1);
  p = B.Part(p).named((m: any) => m.put((k: any) => k.as_native((n: any) => n.set(new Uint8Array([120]))),
    (v: any) => v.as_object((o: any) => o.wires((e: any) => e.sink(hub))))).update();
  assert(same(Reachable.of(p), [p, hub]) && same(Reachable.of(hub), [hub, p]));
  p = B.Part(p).named(new Map()).update(); // the value object is removed, and its entries
  assert(!s0(Plain.ToPlain(store)(Hub, hub)).has("feeds"));
}

## LST-15 · The keyed list classes behave through their protocols

In [ ]:
{
  const key = (value: unknown) => (a: any) => a.as_native((n: any) => n.set(value));
  const seen: unknown[] = [];
  const read = (a: any) => a.as_native((n: any) => seen.push(n.get()));
  const itemKeys = (list: any) => list.pairs((i: any) => i.key(read).value(read));
  // Plain's keyed writers: an array of keys and values (str keys), and a mapping by the keys' text (float keys).
  for (const [name, k1, k2, v1, v2, expected, empty] of [["attrs", "a", "b", 1.5, 2.5, { attrs: [{ key: "b", value: 2.5 }] }, { attrs: [] }],
    ["weights", 1.5, 2.5, "x", "y", { weights: { "2.5": "y" } }, { weights: {} }]] as [string, unknown, unknown, unknown, unknown, object, object][]) {
    const out = new Map<string, any>();
    const writer = new Plain._AnyWriter(out, name, Part.properties.get(name)?.type as S.OfAny.Data, () => new Map(), () => null);
    writer.as_indexed((list) => list.put(key(k1), key(v1)).put(key(k2), key(v1)).put(key(k2), key(v2)));
    seen.length = 0;
    writer.as_indexed((list) => itemKeys(list).at(key(k2), read).items(read)); // a writer reopened reads its own keys
    assert(same(seen, [k1, v1, k2, v2, v2, v1, v2]));
    writer.as_indexed((list) => list.discard(key(k1)));
    assert(equal(out, map(expected)));
    writer.as_indexed((list) => list.item(0, (a) => a.as_native((n) => n.clear())));
    writer.as_indexed((list) => list.put(key(k1), key(v1)).remove(0).clear());
    assert(equal(out, map(empty)));
    for (const [wrong, error, message] of [[(list: any) => list.append(read), TypeError, "a keyed list takes put, not append"],
      [(list: any) => list.at(key(k1), read), LookupError, "the list has no item with this key"],
      [(list: any) => list.put(() => null, read), ValueError, "a key needs a value"]] as [(list: any) => unknown, Function, string][]) {
      raises(error, () => writer.as_indexed(wrong), message);
    }
  }
  {
    const out = new Map<string, any>();
    const writer = new Plain._AnyWriter(out, "cells", Part.properties.get("cells")?.type as S.OfAny.Data, () => new Map(), () => null);
    const cellKey = (r: bigint) => (a: any) => a.as_object((o: any) => o.property("r", (q: any) => q.value(key(r))));
    writer.as_indexed((list) => list.put(cellKey(1n), key(10n)).put(cellKey(2n), key(20n)).put(cellKey(1n), key(11n)));
    assert(equal(out, map({ cells: [{ key: { r: 1n }, value: 11n }, { key: { r: 2n }, value: 20n }] })));
    seen.length = 0;
    writer.as_indexed((list) => list.pairs((i) => i.value(read)));
    assert(same(seen, [11n, 20n]));
  }
  {
    // Plain's positional writer, by key.
    const out = new Map<string, any>();
    const writer = new Plain._AnyWriter(out, "row", Part.properties.get("row")?.type as S.OfAny.Data, () => new Map(), () => null);
    writer.as_indexed((list) => list.put(key(1n), key(10n)).put(key(2n), key(20n)).put(key(1n), key(11n)));
    seen.length = 0;
    writer.as_indexed((list) => itemKeys(list).at(key(2n), read).discard(key(1n)));
    assert(same(seen, [1n, 11n, 2n, 20n, 20n]) && equal(out, map({ row: [20n] })));
    for (const [wrong, error, message] of [[(list: any) => list.at(key(7n), read), LookupError, "the list has no item 7"],
      [(list: any) => list.at(key("1"), read), TypeError, "expected int, got str"],
      [(list: any) => list.at(() => null, read), ValueError, "a key needs a value"]] as [(list: any) => unknown, Function, string][]) {
      raises(error, () => writer.as_indexed(wrong), message);
    }
  }
  // The validation recorder records keys, finds items by them, and keeps every item put.
  const record = new Validators._ListRecord();
  record.put(key("a"), key(1.0)).put(key("b"), key(2.0)).put(key("a"), key(3.0)).append(key(4.0));
  seen.length = 0;
  itemKeys(record).at(key("b"), read);
  assert(same(seen, ["a", 1.0, "b", 2.0, "a", 3.0, null, 4.0, 2.0]));
  record.discard(key("b"));
  assert(same(record.keys, ["a", "a", null]) && same(record.values, [1.0, 3.0, 4.0]));
  raises(LookupError, () => record.at(key("z"), read), "the list has no item with this key");
  seen.length = 0;
  record.at(() => null, read); // no key written: an item appended, which has none
  assert(same(seen, [4.0]));
  // The comparison recorder: a keyed list by key, a positional one by position from its minimum.
  const keyedList = new C.OfIndexed(Part.properties.get("attrs")?.type as S.OfIndexed.Data);
  keyedList.put(key("a"), key(1.0)).put(key("b"), key(2.0)).put(key("a"), key(3.0));
  seen.length = 0;
  itemKeys(keyedList).at(key("b"), read);
  assert(same(seen, ["a", 3.0, "b", 2.0, 2.0]));
  keyedList.discard(key("b")).put(key("c"), (a: any) => a.as_native((n: any) => n.clear())); // no value: left out
  let count = 0;
  keyedList.items(() => count++);
  assert(count === 1);
  for (const [wrong, error, message] of [[() => keyedList.append(read), TypeError, "a keyed list takes put, not append"],
    [() => keyedList.at(key("z"), read), LookupError, "the list has no item with this key"],
    [() => keyedList.put(() => null, read), ValueError, "a key needs a value"]] as [() => unknown, Function, string][]) {
    raises(error, wrong, message);
  }
  const row = new C.OfIndexed(Part.properties.get("row")?.type as S.OfIndexed.Data);
  row.put(key(1n), key(10n)).put(key(2n), key(20n)).put(key(1n), key(11n));
  seen.length = 0;
  itemKeys(row).at(key(2n), read);
  assert(same(seen, [1n, 11n, 2n, 20n, 20n]));
  row.discard(key(1n));
  raises(LookupError, () => row.at(key(3n), read), "the list has no item 3");
  // The reachability collector keeps no items, but follows what is put.
  const nested: any[] = [];
  const collector = new Reachable._Collector((t) => seen.push(t));
  collector.property("named", (p) => p.value((a) => a.as_indexed((list) => nested.push(list))));
  const discarded = nested[0];
  assert(discarded.pairs((x: unknown) => seen.push(x)) === discarded);
  discarded.put((k: unknown) => nested.push(k), (v: any) => v.as_object((o: any) => nested.push(o)));
  assert(nested[2] instanceof Reachable._Collector);
  for (const method of [() => discarded.at(key(new Uint8Array()), read), () => discarded.discard(key(new Uint8Array()))]) {
    raises(LookupError, method, "the list has no item with this key");
  }
}